# 01 Source Review

This is the first notebook for the project. The goal is not to produce a dramatic chart yet. The goal is to learn how to think like a data researcher before trusting any number.

Core question:

> What can each source actually tell us about missing persons in Mexico, and what can it not tell us?

For this topic, source review is not paperwork. It is part of the analysis.

## Learning Goals

By the end of this notebook, you should be able to explain:

- What the official RNPDNO counts.
- Why Data Civica can be easier to start with than the official dashboard.
- Why clandestine grave data is a proxy, not a direct count of missing people.
- Why the impact of madres buscadoras needs a separate event dataset.
- What we should and should not publish for ethical reasons.

In [ ]:
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA = ROOT / "data"
RAW = DATA / "raw"
EXTERNAL = DATA / "external"
PROCESSED = DATA / "processed"

print(f"Project root: {ROOT}")
print(f"External data folder: {EXTERNAL}")

## Source Inventory

A source inventory is a table of what we *think* we can use. Later, we will update this based on what the files actually contain.

Mentor rule: never say "the data says" until you can say who collected it, why they collected it, and what would cause someone to be missing from it.

In [ ]:
sources = pd.DataFrame([
    {
        "source": "RNPDNO official dashboard",
        "owner": "Comision Nacional de Busqueda",
        "url": "https://versionpublicarnpdno.segob.gob.mx/",
        "best_for": "Current official counts by state, municipality, year, sex, and age range",
        "main_risk": "Figures can change retroactively; dashboard has no simple CSV download",
        "counting_status": "Primary source, but must snapshot date/time",
    },
    {
        "source": "Data Civica - Volver a desaparecer",
        "owner": "Data Civica",
        "url": "https://volveradesaparecer.datacivica.org/datos-abiertos",
        "best_for": "Cleaned CSVs and reconstructed historical versions of disappearance registries",
        "main_risk": "Not the live official registry; depends on public versions available at the time",
        "counting_status": "Recommended first dataset for learning and reproducible analysis",
    },
    {
        "source": "Plataforma Ciudadana de Fosas / journalism datasets",
        "owner": "Civil society and investigative journalism groups",
        "url": "https://plataformaciudadanadefosas.org/",
        "best_for": "Clandestine grave discoveries by place and time",
        "main_risk": "Graves, bodies, remains, and fragments are different units; attribution may be unclear",
        "counting_status": "Useful proxy, not a direct count of missing persons",
    },
    {
        "source": "Madres buscadoras event table",
        "owner": "To be built by this project",
        "url": "TBD",
        "best_for": "Documenting collective searches, reported findings, and official confirmation",
        "main_risk": "News coverage bias; safety risks; reports may be incomplete or unconfirmed",
        "counting_status": "Research dataset we will construct carefully",
    },
])

pd.set_option("display.max_colwidth", 120)
sources

## First Research Decisions

Write answers in plain language. These are not final. We revise them as we learn.

1. What do we mean by "missing" in this project?
2. Are we counting people, reports, cases, graves, bodies, remains, or events?
3. What is the earliest year we trust enough to analyze?
4. What geography should be our first unit: state or municipality?
5. What would count as evidence of madres buscadoras impact?
6. What information should we avoid publishing even if we find it?

### Draft Answers

Use this space as a research log.

- Missing definition:
- Unit of analysis:
- Starting year:
- First geography:
- Evidence of impact:
- Do-not-publish rule:

## What Data Files Do We Have Locally?

Before downloading anything, check what is already in the project. This keeps the project reproducible and prevents mystery files.

In [ ]:
data_files = []
for folder in [RAW, EXTERNAL, PROCESSED]:
    folder.mkdir(parents=True, exist_ok=True)
    for path in sorted(folder.glob("**/*")):
        if path.is_file() and path.name != ".gitkeep":
            data_files.append({
                "folder": str(path.parent.relative_to(ROOT)),
                "filename": path.name,
                "size_mb": round(path.stat().st_size / 1_000_000, 3),
            })

pd.DataFrame(data_files) if data_files else "No local data files yet. That is okay. First we review sources, then download deliberately."

## Optional: Load the First CSV

Once you download a CSV into `data/external/`, put the filename below and run the cell.

Recommended first source: Data Civica's reconstructed RNPDNO CSV from `Volver a desaparecer`.

Why this first? Because the official dashboard is important, but for learning we want a stable CSV that we can inspect, rerun, and question.

In [ ]:
csv_filename = ""  # Example: "personas_desaparecidas_2017_2024.csv"

if csv_filename:
    df = pd.read_csv(EXTERNAL / csv_filename)
    print(f"Rows: {len(df):,}")
    print(f"Columns: {len(df.columns):,}")
    display(df.head())
else:
    print("Set csv_filename after placing a CSV in data/external/.")

## Optional: Inspect the Loaded CSV

These checks are boring in the best way. They reveal whether a dataset is usable before we make claims from it.

In [ ]:
if "df" in globals():
    summary = pd.DataFrame({
        "column": df.columns,
        "dtype": [df[col].dtype for col in df.columns],
        "missing_values": [df[col].isna().sum() for col in df.columns],
        "missing_pct": [round(df[col].isna().mean() * 100, 2) for col in df.columns],
        "unique_values": [df[col].nunique(dropna=True) for col in df.columns],
    })
    display(summary)
else:
    print("Load a CSV first.")

## Exit Ticket

Before moving to Notebook 02, answer these in your own words:

1. Which source should be our first analysis source, and why?
2. What number would be irresponsible to publish without caveats?
3. What is one way the official count could undercount or misclassify reality?
4. What is one way media-based data about collectives could be biased?

If you can answer those, you are ready to start counting.